# 01 — Ingest

> **AI-Assisted Development** — This project was built with [Kiro](https://kiro.dev). See `README.md` and `SOURCES.md` for full disclosure.

Fetches all raw data for the **dui-by-state** project and loads it into DuckDB.
Raw files are saved to `data/raw/` unchanged. Nothing is modified here.

### Data sources (all U.S. government / official)
| Table | Source | Notes |
|---|---|---|
| `state_ref` | U.S. Census Bureau Gazetteer | Hand-curated CSV — lat/lng, area, region |
| `fips_codes` | U.S. Census Bureau `state.txt` | Official FIPS codes + abbreviations |
| `population` | Census NST-EST2024 | State pop estimates 2020–2024 |
| `fars_trends` | NHTSA FARS 2015–2024 | Alcohol-impaired fatalities by state/year |
| `fars_2024` | NHTSA FARS 2024 | 2024 state summary |
| `dui_laws_ncsl` | NCSL | DUI criminal status by state |
| `dui_penalties_roadlaw` | roadlawguide.com | First-offense penalties (cross-ref) |
| `dui_penalties_ailawyer` | ailawyer.pro | First-offense penalties (cross-ref) |
| `states` | Derived | Master reference: all of the above joined |
| `_sources` | Metadata | Provenance record for every table |

Full attribution: see `SOURCES.md`.

---
**To re-run ingestion:** execute `scripts/ingest_all.py` from the project root.
This notebook loads the already-ingested data for inspection and verification.

In [ ]:
import sys
sys.path.insert(0, "..")

import duckdb
import pandas as pd
from pathlib import Path
from src.ingest import load_config

cfg = load_config("../config.yaml")
DB  = Path("..") / cfg["settings"]["duckdb_file"]
con = duckdb.connect(str(DB))

print("Project:", cfg["project_name"])
print("DuckDB: ", DB.resolve())
print()
print("Tables in database:")
con.execute("SHOW TABLES").df()

## Source provenance
Every table has an entry in `_sources` recording where the data came from.

In [ ]:
con.execute("SELECT duckdb_table, source_name, license, retrieved FROM _sources").df()

---
## 1. State reference
Hand-curated from the Census Bureau 2024 Gazetteer.
51 rows (50 states + DC). Includes lat/lng centroids, land area, Census region/division.

In [ ]:
state_ref = con.execute("SELECT * FROM state_ref ORDER BY state_fips").df()
print(f"{len(state_ref)} rows  {list(state_ref.columns)}")
state_ref

---
## 2. Population estimates (Census NST-EST2024)
Annual July 1 estimates per state, 2020–2024.

In [ ]:
pop = con.execute("SELECT * FROM population ORDER BY state_fips").df()
print(f"{len(pop)} rows  {list(pop.columns)}")
pop

---
## 3. FARS — alcohol-impaired fatalities (2015–2024)
Source: NHTSA Fatality Analysis Reporting System.

**Methodology note:**
- 2015–2020: `DRUNK_DR > 0` in `accident.csv` flags alcohol-involved crashes.
- 2021–2024: NHTSA restructured the schema. Alcohol involvement derived from
  `drimpair.csv` where `drimpair = 9` (Under the Influence of Alcohol, Drugs or Medication),
  joined to `accident.csv` on `ST_CASE`.
- The 2021+ figures are lower than 2015–2020 because the new schema captures only
  cases explicitly coded — not all alcohol-involved crashes are coded by officers.
  NHTSA's own published totals (using statistical imputation) are higher.

In [ ]:
# National totals by year
national = con.execute("""
    SELECT
        year,
        SUM(total_fatalities)    AS total_fatalities,
        SUM(alcohol_fatalities)  AS alcohol_fatalities,
        ROUND(SUM(alcohol_fatalities) * 100.0 / SUM(total_fatalities), 1) AS pct_alcohol
    FROM fars_trends
    GROUP BY year
    ORDER BY year
""").df()
national

In [ ]:
# 2024 by state — top 15 by alcohol fatality %
con.execute("""
    SELECT state_fips, state_name, total_fatalities, alcohol_fatalities,
           pct_fatalities_alcohol AS pct_alcohol
    FROM fars_2024
    ORDER BY pct_fatalities_alcohol DESC
    LIMIT 15
""").df()

---
## 4. DUI laws & penalties
Three sources scraped and cross-referenced.
Always verify against official state statutes before publishing.

In [ ]:
# NCSL — DUI criminal status (misdemeanor vs felony thresholds)
ncsl = con.execute("SELECT * FROM dui_laws_ncsl").df()
print(f"NCSL: {len(ncsl)} rows")
ncsl.head(10)

In [ ]:
# roadlawguide — penalties: BAC, fines, jail, suspension, IID
roadlaw = con.execute("SELECT * FROM dui_penalties_roadlaw").df()
print(f"roadlawguide: {len(roadlaw)} rows")
roadlaw.head(10)

In [ ]:
# ailawyer — penalties: jail, fine, suspension, IID, lookback, felony threshold
ail = con.execute("SELECT * FROM dui_penalties_ailawyer").df()
print(f"ailawyer: {len(ail)} rows")
ail.head(10)

---
## 5. Master states table
Joins state reference, population, and FARS 2024 summary into one row per state.

In [ ]:
states = con.execute("SELECT * FROM states ORDER BY state_fips").df()
print(f"{len(states)} rows × {len(states.columns)} cols")
print(f"Columns: {list(states.columns)}")
states

---
## Re-running ingestion

If you need to re-pull data from source (e.g., a new year of FARS data):

```bash
cd /path/to/dui-by-state
/opt/anaconda3/envs/data_projects/bin/python scripts/ingest_all.py
```

The script uses `fetch_cached()` — files already on disk won't be re-downloaded.
Delete a specific file from `data/raw/` to force a re-fetch of that source.

---
**Next:** open `02-clean.ipynb` for data cleaning and quality checks.

In [ ]:
con.close()